# Wealth-Based Asset Allocation Optimisation

Optimises a wealth-dependent allocation policy: allocation is a piecewise-linear function of wealth,
parameterised by `WEALTH_NODE_COUNT` evenly-spaced control nodes from `w=0` to `w=2,000,000`.
The policy is interpolated linearly between nodes using `WealthBasedPolicy`.

Initial wealth is drawn from a uniform distribution over [500,000, 1,000,000] so the policy is
optimised simultaneously across the plausible wealth range rather than for a single starting point.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import sys
import math



np.random.seed(42)
torch.manual_seed(42)

sys.path.append('..')

from utils.experiment import (
    SimulationConfig,
    OptimizationResult,
    experiment_filename,
    save_experiment,
    load_experiments,
)

from utils import (
    CholeskyBootstrapReturns,
    BlockBootstrapReturnsLoader,
    WealthBasedPolicy,
    SigmoidWealthPenalty,
    CRRAUtility,
    simulate_wealth_trajectory,
    project_onto_simplex,
)
from utils.spending import *
from utils.allocation import *

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU")


GPU: NVIDIA GeForce RTX 3070
GPU Memory: 8.6 GB


## Configuration Parameters

In [3]:
# ============================================================================
# SIMULATION PARAMETERS
# ============================================================================
N_ASSETS = 3
N_SIMULATIONS = 400_000
SIMULATION_YEARS = 35

# ============================================================================
# POLICY STRUCTURE
# ============================================================================
# Node counts to sweep: 
WEALTH_NODE_COUNTS = [2, 6, 10]

# Wealth grid for the policy nodes (inclusive)
MIN_WEALTH_NODE = 400_000
MAX_WEALTH_NODE = 1_200_000

# ============================================================================
# RETURN SAMPLER
# ============================================================================
RETURN_SAMPLERS = [
    "cholesky",
    "block_bootstrapped",
    "block_bootstrapped_1950",
]

# ============================================================================
# WEALTH & SPENDING PARAMETERS
# ============================================================================
# Initial wealth drawn uniformly from discrete buckets [500_000, 550_000, ..., 1_000_000]
INITIAL_WEALTH_MIN = 500_000
INITIAL_WEALTH_MAX = 1_000_000
WEALTH_STEP = 50_000

DESIRED_SPENDING = EXPENDITURE_GUIDELINES['choices_metro_couple']
SPENDING_DECLINE_RATE = 0.02
CONSUMPTION_FLOOR = EXPENDITURE_GUIDELINES['no_frills_metro_couple']
FLOOR_DECLINE_RATE = 0.00
INCOME_TYPE = "couple"  # "couple" | "single" | "single_sharing" | None

# ============================================================================
# OPTIMIZATION PARAMETERS
# ============================================================================
# Initial risky-asset weights at every node  [bonds_weight, stocks_weight]
INITIAL_NODE_POLICY = [0.05, 0.9]

OPTIMIZER_TYPE = "SGD"
INITIAL_LR = .5
MIN_LR = 1e-4
MAX_ITERATIONS = 10_000
MOMENTUM = 0.9

# Learning-rate schedule
LR_DECAY_FACTOR = 0.1
LR_PATIENCE = 80
LR_THRESHOLD = 1e-4

# Early stopping
STOPPING_PATIENCE = 300

# Progress reporting
PRINT_EVERY = 5
HISTORY_SAVE_FREQUENCY = 20

# ============================================================================
# OBJECTIVE FUNCTION PARAMETERS
# ============================================================================
WEALTH_PENALTY_STEEPNESS = 1e-3


## Load Data

In [4]:
tax_rates = np.loadtxt("../Data/IID Data/Final/tax_rates.csv", delimiter=",", skiprows=1)

return_sampler_loaders = {}
for sampler_name in RETURN_SAMPLERS:
    if sampler_name == "cholesky":
        exp_returns = np.loadtxt("../Data/IID Data/Final/expected_returns.csv", delimiter=",", skiprows=1)
        cov_matrix = np.loadtxt(
            "../Data/IID Data/Final/covariance.csv", delimiter=",", skiprows=1, usecols=range(1, N_ASSETS + 2)
        )
        return_sampler_loaders[sampler_name] = CholeskyBootstrapReturns(exp_returns, cov_matrix)
    elif sampler_name in {"block_bootstrapped", "block_bootstrapped_1950"}:
        return_sampler_loaders[sampler_name] = BlockBootstrapReturnsLoader(
            f"../Data/Returns/Final/{sampler_name}.npy"
        )
    else:
        raise ValueError(f"Unknown RETURN_SAMPLER: {sampler_name}")


## Optimisation

In [5]:
completed_runs = set()
try:
    existing_experiments = load_experiments(results_dir="Results")
    for payload in existing_experiments:
        cfg = payload["config"]
        completed_runs.add((cfg.RETURN_SAMPLER, int(cfg.WEALTH_NODE_COUNT)))
    print(f"Loaded {len(completed_runs)} completed runs from Results/.")
except FileNotFoundError:
    print("Results/ not found yet - all runs will be executed.")
except Exception as exc:
    print(f"Could not read existing Results ({exc}); proceeding without skip cache.")


Loaded 0 completed runs from Results/.


In [6]:
from IPython.display import clear_output
from wakepy import keep

wealth_penalty = SigmoidWealthPenalty(steepness=WEALTH_PENALTY_STEEPNESS)

# Build bucketed initial wealth tensor: equal numbers of sims at each
# discrete level [INITIAL_WEALTH_MIN, ..., INITIAL_WEALTH_MAX] in steps of WEALTH_STEP.
wealth_levels = torch.arange(
    INITIAL_WEALTH_MIN, INITIAL_WEALTH_MAX + WEALTH_STEP, WEALTH_STEP, dtype=torch.float32
)
n_buckets = len(wealth_levels)
repeats = N_SIMULATIONS // n_buckets
remainder = N_SIMULATIONS % n_buckets
initial_wealth_tensor = torch.cat([
    wealth_levels.repeat(repeats),
    wealth_levels[:remainder],
]).to(DEVICE)
print(f"Initial wealth buckets: {wealth_levels.numpy()} ({n_buckets} levels, ~{repeats:,} sims each)")

with keep.running():
    for RETURN_SAMPLER in RETURN_SAMPLERS:
        sampler = return_sampler_loaders[RETURN_SAMPLER]
        returns_sample, cumulative_inflation_sample = sampler.generate(N_SIMULATIONS, SIMULATION_YEARS)

        for WEALTH_NODE_COUNT in WEALTH_NODE_COUNTS:
            run_key = (RETURN_SAMPLER, int(WEALTH_NODE_COUNT))
            if run_key in completed_runs:
                print(
                    f"Skipping completed run: sampler={RETURN_SAMPLER}, "
                    f"WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                )
                continue

            # Wealth nodes: logarithmically spaced from MIN_WEALTH_NODE to MAX_WEALTH_NODE.
            # Denser coverage at lower wealth levels where policy sensitivity is higher.
            wealth_nodes = torch.logspace(
                math.log10(MIN_WEALTH_NODE), math.log10(MAX_WEALTH_NODE), WEALTH_NODE_COUNT
            )

            print(f"\n{'='*70}")
            print(f"STARTING COMBINATION: sampler={RETURN_SAMPLER}, WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}")
            print(f"Wealth nodes: {wealth_nodes.numpy()}")
            print(f"{'='*70}")

            returns = torch.tensor(returns_sample - tax_rates, device=DEVICE)
            cumulative_inflation = torch.tensor(cumulative_inflation_sample, device=DEVICE)

            sim_config = SimulationConfig(
                N_ASSETS=N_ASSETS,
                N_SIMULATIONS=N_SIMULATIONS,
                SIMULATION_YEARS=SIMULATION_YEARS,
                RETURN_SAMPLER=RETURN_SAMPLER,
                INITIAL_WEALTH=None,  # Not fixed; distributed across wealth buckets
                DESIRED_SPENDING=DESIRED_SPENDING,
                SPENDING_DECLINE_RATE=SPENDING_DECLINE_RATE,
                CONSUMPTION_FLOOR=CONSUMPTION_FLOOR,
                FLOOR_DECLINE_RATE=FLOOR_DECLINE_RATE,
                INCOME_TYPE=INCOME_TYPE,
                INITIAL_POLICY=INITIAL_NODE_POLICY,
                OPTIMIZER_TYPE=OPTIMIZER_TYPE,
                TIME_NODE_COUNT=1,
                WEALTH_NODE_COUNT=WEALTH_NODE_COUNT,
            )

            desired_spending_pol = DecliningRealSpending(DESIRED_SPENDING, decline_rate=SPENDING_DECLINE_RATE)
            consumption_floor_pol = DecliningRealFloor(init_floor=CONSUMPTION_FLOOR, decline_rate=FLOOR_DECLINE_RATE)
            income = NZSuper(INCOME_TYPE)
            spending_policy = SpendingPolicy(
                spending=desired_spending_pol,
                floor=consumption_floor_pol,
                income=income,
            )

            allocation_policy = WealthBasedPolicy(N_ASSETS, N_SIMULATIONS, wealth_nodes.clone(), DEVICE)

            # Policy tensor: shape (WEALTH_NODE_COUNT, N_ASSETS-1)
            # Each row is the [bonds, stocks] allocation at that wealth node.
            policy = torch.tensor(
                [INITIAL_NODE_POLICY] * WEALTH_NODE_COUNT,
                device=DEVICE, dtype=torch.float32, requires_grad=True,
            )

            if OPTIMIZER_TYPE.upper() == "ADAM":
                optimizer = torch.optim.Adam([policy], lr=INITIAL_LR)
            else:
                optimizer = torch.optim.SGD([policy], lr=INITIAL_LR, momentum=MOMENTUM)

            cost_history = []
            policy_history = []
            best_cost = float('inf')
            best_policy = policy.data.clone()
            iterations_without_improvement = 0
            iterations_without_lr_improvement = 0

            for i in range(MAX_ITERATIONS):
                optimizer.zero_grad()

                wealth, consumption = simulate_wealth_trajectory(
                    returns=returns,
                    cumulative_inflation=cumulative_inflation,
                    allocation_policy=allocation_policy,
                    spending_policy=spending_policy,
                    initial_wealth=initial_wealth_tensor,
                    policy_settings=policy,
                )

                cost = wealth_penalty.evaluate(wealth, consumption)
                cost.backward()
                optimizer.step()

                # project_onto_simplex is vectorised: projects each row independently
                with torch.no_grad():
                    policy.data = project_onto_simplex(policy.data).clamp(0, 1)

                cost_item = cost.item()
                cost_history.append(cost_item)

                if (i + 1) % HISTORY_SAVE_FREQUENCY == 0:
                    policy_history.append(policy.data.cpu().numpy().copy())

                if cost_item < best_cost:
                    best_cost = cost_item
                    best_policy = policy.detach().clone().cpu()
                    iterations_without_improvement = 0
                    iterations_without_lr_improvement = 0
                else:
                    iterations_without_improvement += 1
                    iterations_without_lr_improvement += 1

                if iterations_without_improvement >= STOPPING_PATIENCE:
                    print(f"\n{'='*70}")
                    print(f"EARLY STOPPING at iteration {i + 1}")
                    print(f"No improvement in {STOPPING_PATIENCE} iterations")
                    print(f"{'='*70}")
                    break

                if iterations_without_lr_improvement >= LR_PATIENCE:
                    stop = False
                    for param_group in optimizer.param_groups:
                        old_lr = param_group['lr']
                        new_lr = old_lr * LR_DECAY_FACTOR
                        param_group['lr'] = new_lr
                        if new_lr < MIN_LR:
                            print(f"\n{'='*70}")
                            print(f"LR below minimum threshold at iteration {i + 1}: {new_lr:.6f} < {MIN_LR:.6f}")
                            print(f"Stopping optimization.")
                            print(f"{'='*70}")
                            stop = True
                            break
                    if stop:
                        break
                    print(f"LR reduced at iteration {i+1}: {old_lr:.6f} -> {new_lr:.6f}")
                    iterations_without_lr_improvement = 0

                    with torch.no_grad():
                        policy.copy_(best_policy.to(policy.device))

                if (i + 1) % PRINT_EVERY == 0 or i == 0:
                    clear_output(wait=True)
                    current_lr = optimizer.param_groups[0]['lr']
                    cost_change = cost_history[-1] - cost_history[-2] if len(cost_history) > 1 else 0

                    print(f"{'='*70}")
                    print(
                        f"OPTIMISING: sampler={RETURN_SAMPLER} | "
                        f"WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                    )
                    print(f"Iteration {i+1:,}/{MAX_ITERATIONS:,} ({(i+1)/MAX_ITERATIONS*100:.1f}%)  |  LR: {current_lr:.6f}")
                    print(f"{'='*70}")
                    print(f"Cost:                      {cost_item:.6f}")
                    print(f"Cost change:               {cost_change:.8f}")
                    print(f"Best Cost:                 {best_cost:.6f}")
                    print(f"Iterations w/o improve:    {iterations_without_improvement}/{STOPPING_PATIENCE}")
                    print(f"{'-'*70}")
                    for node_idx in range(WEALTH_NODE_COUNT):
                        alloc = policy.data[node_idx].cpu().numpy()
                        cash = 1.0 - alloc.sum()
                        node_w = wealth_nodes[node_idx].item()
                        print(
                            f"  Node {node_idx} (w=${node_w:,.0f}):  "
                            f"cash={cash:.1%}  bonds={alloc[0]:.1%}  stocks={alloc[1]:.1%}"
                        )
                    print(f"{'-'*70}")
                    print(f"Mean consumption:          ${consumption.mean().item():,.0f}")
                    print(f"Mean terminal wealth:      ${wealth[:, -1].mean().item():,.0f}")
                    print(f"Bankruptcy rate:           {(wealth[:, -1] == 0).sum().item() / N_SIMULATIONS:.2%}")
                    bankruptcy_density = (wealth == 0).sum().item() / (N_SIMULATIONS * SIMULATION_YEARS)
                    floor_t = consumption_floor_pol.calculate_tensor(
                        wealth=wealth, cumulative_inflation=cumulative_inflation
                    )
                    impoverishment_density = (consumption < floor_t).sum().item() / (N_SIMULATIONS * SIMULATION_YEARS)
                    print(f"Impoverishment density:    {impoverishment_density:.4%}")
                    print(f"Bankruptcy density:        {bankruptcy_density:.4%}")
                    print(f"{'='*70}")

            with torch.no_grad():
                policy.copy_(best_policy.to(policy.device))

            policy_history = np.array(policy_history)
            cost_history = np.array(cost_history)

            print(f"\n{'='*70}")
            print(
                f"OPTIMISATION COMPLETE - sampler={RETURN_SAMPLER} "
                f"| WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
            )
            print(f"Best cost: {best_cost:.6f}  |  Iterations: {len(cost_history):,}")
            for node_idx in range(WEALTH_NODE_COUNT):
                alloc = best_policy[node_idx].cpu().numpy()
                cash = 1.0 - alloc.sum()
                node_w = wealth_nodes[node_idx].item()
                print(
                    f"  Node {node_idx} (w=${node_w:,.0f}):  "
                    f"cash={cash:.1%}  bonds={alloc[0]:.1%}  stocks={alloc[1]:.1%}"
                )
            print(f"{'='*70}")

            final_wealth, final_consumption = simulate_wealth_trajectory(
                returns=returns,
                cumulative_inflation=cumulative_inflation,
                allocation_policy=allocation_policy,
                spending_policy=spending_policy,
                initial_wealth=initial_wealth_tensor,
                policy_settings=policy,
            )

            result = OptimizationResult(
                best_policy=best_policy.detach().cpu().numpy(),
                best_utility=-best_cost,
                policy_history=policy_history,
                cost_history=cost_history,
                wealth_simulated=final_wealth.detach().cpu().numpy(),
                consumption_simulated=final_consumption.detach().cpu().numpy(),
                cumulative_inflation=cumulative_inflation.detach().cpu().numpy(),
                wealth_nodes=wealth_nodes.detach().cpu().numpy(),
            )
            save_experiment(sim_config, result)
            completed_runs.add(run_key)


OPTIMISING: sampler=block_bootstrapped_1950 | WEALTH_NODE_COUNT=10
Iteration 475/10,000 (4.8%)  |  LR: 0.000500
Cost:                      -0.963091
Cost change:               0.00000000
Best Cost:                 -0.963091
Iterations w/o improve:    295/300
----------------------------------------------------------------------
  Node 0 (w=$400,000):  cash=0.0%  bonds=0.0%  stocks=100.0%
  Node 1 (w=$451,932):  cash=0.4%  bonds=7.9%  stocks=91.7%
  Node 2 (w=$510,607):  cash=35.8%  bonds=0.0%  stocks=64.2%
  Node 3 (w=$576,900):  cash=38.4%  bonds=0.0%  stocks=61.6%
  Node 4 (w=$651,799):  cash=57.0%  bonds=0.0%  stocks=43.0%
  Node 5 (w=$736,423):  cash=52.5%  bonds=0.0%  stocks=47.5%
  Node 6 (w=$832,033):  cash=67.3%  bonds=0.0%  stocks=32.7%
  Node 7 (w=$940,057):  cash=59.7%  bonds=0.0%  stocks=40.3%
  Node 8 (w=$1,062,106):  cash=39.9%  bonds=0.0%  stocks=60.1%
  Node 9 (w=$1,200,000):  cash=33.6%  bonds=0.0%  stocks=66.4%
---------------------------------------------------------